# 02 - Data Cleaning Pipeline

**Goal**: Transform raw postings into a clean, analysis-ready dataset saved as Parquet.
This notebook documents every cleaning decision and its rationale.

**Learning concepts**: Data wrangling, null handling strategies, timestamp parsing,
salary normalization, Parquet format vs CSV.

**Why Parquet?**
- CSV: 493MB, slow to load, no type info, everything is a string on disk
- Parquet: ~100MB (columnar compression), fast to load, preserves dtypes, industry standard

---

In [ ]:
import pandas as pd
import numpy as np
from loguru import logger

from talentlens.config import (
    POSTINGS_CSV, PROCESSED_DIR, POSTINGS_CLEAN_PARQUET,
    SALARIES_CSV, HOURS_PER_YEAR,
)
from talentlens.dataset import load_raw_postings, clean_postings, load_secondary_tables

pd.set_option('display.max_columns', None)

## 1. Load the Full Raw Dataset

We load all 3.38M rows. This may take 1-2 minutes depending on your machine.

> **For development**: Set `nrows=50000` to work with a smaller sample first.

In [ ]:
import time

# Set nrows=50000 for faster development; nrows=None for full dataset
start = time.time()
df_raw = load_raw_postings(nrows=None)
elapsed = time.time() - start

print(f"Raw shape: {df_raw.shape}")
print(f"Memory: {df_raw.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"Load time: {elapsed:.1f} seconds")

## 2. Understand What We're Dropping (and Why)

Before cleaning, let's see what's missing so we can justify our decisions.

In [ ]:
# We only profile the columns relevant to our 4 research themes — not all 27.
# Why these? Each maps to a research question:
#   description → NLP text for RAG and classification
#   med_salary  → Theme 1 (salary prediction)
#   applies, views → Theme 2 (ghost jobs) & Theme 4 (engagement)
#   formatted_experience_level → Theme 3 (entry-level paradox)
#   listed_time, closed_time → Theme 2 (how long a job stays open)
critical_cols = ['description', 'title', 'med_salary', 'min_salary', 'max_salary',
                 'remote_allowed', 'formatted_experience_level', 'skills_desc',
                 'applies', 'views', 'listed_time', 'closed_time']

null_report = pd.DataFrame({
    'null_count': df_raw[critical_cols].isna().sum(),
    'null_pct': (df_raw[critical_cols].isna().mean() * 100).round(1),
    'non_null': df_raw[critical_cols].count(),
})
null_report.sort_values('null_pct', ascending=False)

In [ ]:
# How many duplicate job_ids?
# Good practice: ALWAYS check for duplicates, even if you expect none.
# In production, data arrives dirty in unpredictable ways — a duplicate
# check costs nothing and would catch a real problem instantly.
dupes = df_raw.duplicated(subset=['job_id'], keep=False)
print(f"Duplicate job_id rows: {dupes.sum():,} ({dupes.mean()*100:.1f}%)")
print(f"Unique job_ids: {df_raw['job_id'].nunique():,}")

if dupes.sum() == 0:
    print("\nNo duplicates found — this dataset is well-curated (typical for Kaggle).")
    print("In real-world pipelines, you'd often find 5-20% duplicates here.")

## 3. Run the Cleaning Pipeline

Our `clean_postings()` function (from `talentlens/dataset.py`) performs:

1. **Drop null descriptions** — can't do NLP without text
2. **Drop null titles** — need titles for classification
3. **Deduplicate on job_id** — keep first occurrence
4. **Parse timestamps** — convert Unix ms → datetime
5. **Normalize salaries** — hourly × 2,080 → yearly
6. **Create derived columns** — `is_remote`, `days_open`, `experience_level`

In [ ]:
# Track counts BEFORE each step so we can report per-step drops
count_before = len(df_raw)
count_after_desc = len(df_raw.dropna(subset=["description"]))
count_after_title = len(df_raw.dropna(subset=["description"]).dropna(subset=["title"]))

df_clean = clean_postings(df_raw)

print(f"\n{'='*55}")
print(f"  CLEANING SUMMARY")
print(f"{'='*55}")
print(f"  Raw rows:              {count_before:>12,}")
print(f"  After drop null desc:  {count_after_desc:>12,}  (-{count_before - count_after_desc:,})")
print(f"  After drop null title: {count_after_title:>12,}  (-{count_after_desc - count_after_title:,})")
print(f"  After deduplication:   {len(df_clean):>12,}  (-{count_after_title - len(df_clean):,})")
print(f"{'='*55}")
print(f"  Total dropped: {count_before - len(df_clean):,} rows ({(1 - len(df_clean)/count_before)*100:.2f}%)")
print(f"{'='*55}")
print(f"\n>>> UPDATE YOUR RESUME WITH THIS NUMBER: {len(df_clean):,} job postings <<<")

## 4. Validate the Cleaned Data

In [ ]:
# Sanity checks
assert df_clean['description'].isna().sum() == 0, "Null descriptions found!"
assert df_clean['title'].isna().sum() == 0, "Null titles found!"
assert df_clean['job_id'].is_unique, "Duplicate job_ids found!"
assert 'is_remote' in df_clean.columns, "Missing is_remote column!"
assert 'experience_level' in df_clean.columns, "Missing experience_level column!"
assert 'med_salary_yearly' in df_clean.columns, "Missing med_salary_yearly column!"

print("All validation checks passed!")
print(f"\nCleaned columns ({len(df_clean.columns)}):")
for col in sorted(df_clean.columns):
    print(f"  {col}: {df_clean[col].dtype}")

## 5. Inspect Key Derived Columns

In [ ]:
# Experience level distribution
print("=== Experience Levels ===")
print(df_clean['experience_level'].value_counts())

print("\n=== Remote Work ===")
print(df_clean['is_remote'].value_counts())

# Salary stats with explicit interpretation
print("\n=== Salary Coverage ===")
has_salary = df_clean['med_salary_yearly'].notna()
print(f"Postings with salary: {has_salary.sum():,} ({has_salary.mean()*100:.1f}%)")
print(f"Postings WITHOUT salary: {(~has_salary).sum():,}")
print("  → Most postings don't list salary. For Theme 1 (salary prediction),")
print("    we'll only train on the subset that has salary data.")

print("\n=== Salary Stats (yearly, where available) ===")
salary = df_clean['med_salary_yearly'].dropna()
print(f"  Count:   {len(salary):,}")
print(f"  Min:     ${salary.min():>12,.0f}")
print(f"  25th %:  ${salary.quantile(0.25):>12,.0f}")
print(f"  Median:  ${salary.median():>12,.0f}  ← typical job posting salary")
print(f"  75th %:  ${salary.quantile(0.75):>12,.0f}")
print(f"  Max:     ${salary.max():>12,.0f}")
print(f"  Mean:    ${salary.mean():>12,.0f}")

print("\n=== Days Open Stats ===")
if 'days_open' in df_clean.columns:
    days = df_clean['days_open'].dropna()
    print(f"  Count:   {len(days):,}")
    print(f"  Min:     {days.min():.0f} days")
    print(f"  Median:  {days.median():.0f} days")
    print(f"  Mean:    {days.mean():.1f} days")
    print(f"  Max:     {days.max():.0f} days")
    print(f"\n  → Median is 0: most postings close on the same day they're listed.")
    print(f"    The long tail (>60 days open) is where ghost job candidates live.")
    print(f"    Postings open > 60 days: {(days > 60).sum():,} ({(days > 60).mean()*100:.1f}%)")

## 6. Save the Cleaned Dataset

In [ ]:
# Ensure output directory exists
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save as Parquet (much smaller and faster than CSV)
df_clean.to_parquet(POSTINGS_CLEAN_PARQUET, index=False)

# Compare file sizes
import os
csv_size = os.path.getsize(POSTINGS_CSV) / 1e6
parquet_size = os.path.getsize(POSTINGS_CLEAN_PARQUET) / 1e6

print(f"Raw CSV size:       {csv_size:>8.1f} MB")
print(f"Clean Parquet size: {parquet_size:>8.1f} MB")
print(f"Compression ratio:  {csv_size/parquet_size:.1f}x smaller")

# Verify we can reload it
df_verify = pd.read_parquet(POSTINGS_CLEAN_PARQUET)
assert len(df_verify) == len(df_clean), "Row count mismatch!"
print(f"\nVerified: {len(df_verify):,} rows saved and reloaded successfully.")

## Cleaning Decisions Log

| Decision | Rationale | Rows Affected |
|----------|-----------|---------------|
| Drop null descriptions | Can't do NLP/RAG without text | *(fill in)* |
| Drop null titles | Need titles for job classification | *(fill in)* |
| Deduplicate on job_id | Same posting listed multiple times | *(fill in)* |
| Convert hourly → yearly salary | Need consistent units for comparison | N/A (transform) |
| Parse timestamps to datetime | Enable time-based analysis | N/A (transform) |

---

**→ Next notebook**: `03-mp-exploratory-data-analysis.ipynb`